<a href="https://colab.research.google.com/github/narakrm/northstar_databases_analytics/blob/main/Section7_MongoDB_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 7 — MongoDB Development
**NorthStar Urban Mobility and Logistics — Databases and Analytics Assignment**

This notebook implements the MongoDB Atlas NoSQL database for NorthStar using PyMongo.

**Contents:**
1. Install dependencies and connect to Atlas
2. Load and clean all CSV files
3. Build and insert `customer_cases` collection (650 documents)
4. Build and insert `operational_events` collection (640 documents)
5. Build and insert `fleet_health` collection (120 documents)
6. CRUD operations — Create, Read, Update, Delete
7. Aggregation pipelines

---
> **Setup:** Upload all 9 NorthStar CSV files to `/content/` before running.  
> **Atlas URI:** Store your MongoDB Atlas connection string in Colab Secrets as `MONGO_URI`  
> (Runtime → Manage Secrets → Add `MONGO_URI`)

## Cell 1 — Install PyMongo

In [ ]:
!pip install pymongo --quiet

import pandas as pd
import numpy as np
from pymongo import MongoClient, ASCENDING
from pymongo.errors import ConnectionFailure, BulkWriteError
from google.colab import userdata
from collections import defaultdict
import pprint
import json

print('PyMongo imported successfully')

## Cell 2 — Connect to MongoDB Atlas

In [ ]:
# Retrieve URI from Colab Secrets
MONGO_URI = userdata.get('MONGO_URI')

# Connect
client = MongoClient(MONGO_URI)

# Test connection
try:
    client.admin.command('ping')
    print('Connected to MongoDB Atlas successfully')
    print('Server version:', client.server_info()['version'])
except ConnectionFailure as e:
    print('Connection failed:', e)

# Select database and collections
db                 = client['northstar_db']
customer_cases     = db['customer_cases']
operational_events = db['operational_events']
fleet_health       = db['fleet_health']

print('Database: northstar_db')
print('Collections ready:', ['customer_cases', 'operational_events', 'fleet_health'])

## Cell 3 — Load and clean all CSV files

In [ ]:
data_path = '/content/'

orders_df     = pd.read_csv(data_path + 'orders.csv')
deliveries_df = pd.read_csv(data_path + 'deliveries.csv')
complaints_df = pd.read_csv(data_path + 'complaints.csv')
app_events_df = pd.read_csv(data_path + 'app_events.csv')
customers_df  = pd.read_csv(data_path + 'customers.csv')
vehicles_df   = pd.read_csv(data_path + 'vehicles.csv')
incidents_df  = pd.read_csv(data_path + 'incidents.csv')

# Zone standardisation — 16 raw variants -> 7 canonical labels
zone_map = {
    'NORTH': 'North', 'north': 'North',
    'SOUTH': 'South', 'EAST': 'East',
    'WEST': 'West',   'CENTRAL': 'Central', 'CTR': 'Central', 'Ctr': 'Central',
    'AIRPORT': 'Airport',
    'RIVERSIDE': 'Riverside', 'RiverSide': 'Riverside'
}

for df, cols in [
    (customers_df,  ['home_zone']),
    (orders_df,     ['pickup_zone', 'dropoff_zone']),
    (app_events_df, ['zone_context']),
    (vehicles_df,   ['assigned_zone']),
    (deliveries_df, [])
]:
    for col in cols:
        df[col] = df[col].str.strip().replace(zone_map)

# Numeric type conversions
orders_df['order_value']                         = pd.to_numeric(orders_df['order_value'], errors='coerce')
deliveries_df['fuel_or_charge_cost']             = pd.to_numeric(deliveries_df['fuel_or_charge_cost'], errors='coerce')
deliveries_df['manual_route_override_count']     = pd.to_numeric(deliveries_df['manual_route_override_count'], errors='coerce')
deliveries_df['customer_rating_post_delivery']   = pd.to_numeric(deliveries_df['customer_rating_post_delivery'], errors='coerce')
customers_df['loyalty_score']                    = pd.to_numeric(customers_df['loyalty_score'], errors='coerce')
customers_df['app_engagement_score']             = pd.to_numeric(customers_df['app_engagement_score'], errors='coerce')
complaints_df['compensation_amount']             = pd.to_numeric(complaints_df['compensation_amount'], errors='coerce')
complaints_df['resolution_days']                 = pd.to_numeric(complaints_df['resolution_days'], errors='coerce')
vehicles_df['battery_health_pct']                = pd.to_numeric(vehicles_df['battery_health_pct'], errors='coerce')
vehicles_df['odometer_km']                       = pd.to_numeric(vehicles_df['odometer_km'], errors='coerce')
app_events_df['api_latency_ms']                  = pd.to_numeric(app_events_df['api_latency_ms'], errors='coerce')
incidents_df['resolved_hours']                   = pd.to_numeric(incidents_df['resolved_hours'], errors='coerce')

print('All files loaded and cleaned')
print(f'  customers: {len(customers_df)} | orders: {len(orders_df)} | deliveries: {len(deliveries_df)}')
print(f'  complaints: {len(complaints_df)} | app_events: {len(app_events_df)}')
print(f'  vehicles: {len(vehicles_df)} | incidents: {len(incidents_df)}')

---
## Cell 4 — Build and insert `customer_cases` collection

Each document embeds a customer's order history (with delivery outcome nested inside each order), complaint records, and app events.  
This produces the integrated customer view the board requested — available in a single query.

In [ ]:
# Drop existing collection to avoid duplicates on re-run
customer_cases.drop()

# Build lookup maps
delivery_map    = deliveries_df.set_index('order_id').to_dict('index')
complaints_map  = complaints_df.groupby('customer_id').apply(
                      lambda x: x.to_dict('records')).to_dict()
events_map      = app_events_df.groupby('customer_id').apply(
                      lambda x: x.to_dict('records')).to_dict()
orders_map      = orders_df.groupby('customer_id').apply(
                      lambda x: x.to_dict('records')).to_dict()

def clean_dict(d):
    """Replace NaN with None for MongoDB compatibility."""
    if d is None:
        return None
    return {k: (None if (isinstance(v, float) and np.isnan(v)) else v)
            for k, v in d.items()}

def build_customer_doc(row):
    cid = row['customer_id']

    # Build orders with embedded delivery
    cust_orders = []
    for o in orders_map.get(cid, []):
        o_clean = clean_dict(o)
        oid = o_clean.get('order_id')
        delivery = clean_dict(delivery_map.get(oid)) if oid in delivery_map else None
        o_clean['delivery'] = delivery
        cust_orders.append(o_clean)

    # Clean complaints and events
    cust_complaints = [clean_dict(c) for c in complaints_map.get(cid, [])]
    cust_events     = [clean_dict(e) for e in events_map.get(cid, [])]

    return {
        'customer_id':          cid,
        'age':                  int(row['age']) if pd.notna(row['age']) else None,
        'home_zone':            row['home_zone'],
        'customer_type':        row['customer_type'],
        'signup_date':          row['signup_date'],
        'loyalty_score':        float(row['loyalty_score']) if pd.notna(row['loyalty_score']) else None,
        'app_engagement_score': float(row['app_engagement_score']) if pd.notna(row['app_engagement_score']) else None,
        'preferred_channel':    row['preferred_channel'],
        'account_status':       row['account_status'],
        'orders':               cust_orders,
        'complaints':           cust_complaints,
        'app_events':           cust_events
    }

# Build all documents
customer_docs = [build_customer_doc(row) for _, row in customers_df.iterrows()]

# Insert
result = customer_cases.insert_many(customer_docs)
print(f'Inserted {len(result.inserted_ids)} documents into customer_cases')

# Verify with a sample
sample = customer_cases.find_one({'customer_id': 'C0182'})
print(f"\nSample — C0182: {sample['customer_type']}, {sample['home_zone']} zone")
print(f"  Orders: {len(sample['orders'])} | Complaints: {len(sample['complaints'])} | Events: {len(sample['app_events'])}")

---
## Cell 5 — Build and insert `operational_events` collection

Reshapes flat app_events rows into session-based documents with an ordered events array.  
Preserves the interaction sequence context that is lost when events are stored as individual rows.

In [ ]:
operational_events.drop()

# Build customer lookup for enrichment
cust_lookup = customers_df.set_index('customer_id')[['customer_type','home_zone']].to_dict('index')

# Group events by session_id
session_groups = app_events_df.groupby('session_id')

session_docs = []
for session_id, group in session_groups:
    group_sorted = group.sort_values('event_timestamp')
    cid = group_sorted.iloc[0]['customer_id']
    cust_info = cust_lookup.get(cid, {})

    events = []
    for _, row in group_sorted.iterrows():
        events.append({
            'event_id':        row['event_id'],
            'event_type':      row['event_type'],
            'event_timestamp': row['event_timestamp'],
            'order_id':        row['order_id'] if pd.notna(row['order_id']) and row['order_id'] != '' else None,
            'zone_context':    row['zone_context'],
            'api_latency_ms':  int(row['api_latency_ms']) if pd.notna(row['api_latency_ms']) else None,
            'success_flag':    int(row['success_flag']) if pd.notna(row['success_flag']) else None
        })

    session_docs.append({
        'session_id':    session_id,
        'customer_id':   cid,
        'customer_type': cust_info.get('customer_type', None),
        'home_zone':     cust_info.get('home_zone', None),
        'device_type':   group_sorted.iloc[0]['device_type'],
        'session_start': group_sorted.iloc[0]['event_timestamp'],
        'session_end':   group_sorted.iloc[-1]['event_timestamp'],
        'event_count':   len(events),
        'events':        events
    })

result = operational_events.insert_many(session_docs)
print(f'Inserted {len(result.inserted_ids)} documents into operational_events')

# Verify
sample_sess = operational_events.find_one({'customer_id': 'C0144'})
if sample_sess:
    print(f"\nSample session {sample_sess['session_id']}: {sample_sess['event_count']} event(s)")
    for e in sample_sess['events']:
        print(f"  {e['event_type']} @ {e['event_timestamp']}")

---
## Cell 6 — Build and insert `fleet_health` collection

Merges vehicles, incidents, and recent deliveries into per-vehicle documents.  
Enables proactive fault detection by co-locating battery health, incident history, and route assignments.

In [ ]:
fleet_health.drop()

# Build delivery lookup by vehicle_id
deliveries_by_vehicle = deliveries_df.groupby('vehicle_id').apply(
    lambda x: x.sort_values('dispatch_time', ascending=False).head(5).to_dict('records')
).to_dict()

# Build incident lookup by delivery_id
incidents_by_delivery = incidents_df.groupby('delivery_id').apply(
    lambda x: x.to_dict('records')
).to_dict()

def build_fleet_doc(row):
    vid = row['vehicle_id']

    # Recent deliveries (up to 5)
    recent_deliveries = []
    for d in deliveries_by_vehicle.get(vid, []):
        recent_deliveries.append({
            'delivery_id':      d.get('delivery_id'),
            'order_id':         d.get('order_id'),
            'dispatch_time':    d.get('dispatch_time'),
            'delivery_status':  d.get('delivery_status'),
            'route_distance_km': float(d['route_distance_km']) if pd.notna(d.get('route_distance_km')) else None,
            'driver_id':        d.get('driver_id')
        })

    # Incident history — via delivery_ids for this vehicle
    incident_history = []
    vehicle_delivery_ids = [d.get('delivery_id') for d in deliveries_by_vehicle.get(vid, [])]
    for did in vehicle_delivery_ids:
        for inc in incidents_by_delivery.get(did, []):
            incident_history.append({
                'incident_id':       inc.get('incident_id'),
                'incident_type':     inc.get('incident_type'),
                'reported_at':       inc.get('reported_at'),
                'severity':          inc.get('severity'),
                'resolution_status': inc.get('resolution_status'),
                'resolved_hours':    float(inc['resolved_hours']) if pd.notna(inc.get('resolved_hours')) else None
            })

    return {
        'vehicle_id':         vid,
        'vehicle_type':       row['vehicle_type'],
        'assigned_zone':      row['assigned_zone'],
        'commission_date':    row['commission_date'],
        'battery_health_pct': float(row['battery_health_pct']) if pd.notna(row['battery_health_pct']) else None,
        'odometer_km':        int(row['odometer_km']) if pd.notna(row['odometer_km']) else None,
        'maintenance_status': row['maintenance_status'],
        'telematics_version': row['telematics_version'],
        'recent_deliveries':  recent_deliveries,
        'incident_history':   incident_history
    }

fleet_docs = [build_fleet_doc(row) for _, row in vehicles_df.iterrows()]
result = fleet_health.insert_many(fleet_docs)
print(f'Inserted {len(result.inserted_ids)} documents into fleet_health')

# Verify V002
v002 = fleet_health.find_one({'vehicle_id': 'V002'})
print(f"\nV002: {v002['maintenance_status']}, battery {v002['battery_health_pct']}%, zone {v002['assigned_zone']}")
print(f"  Recent deliveries: {len(v002['recent_deliveries'])} | Incidents: {len(v002['incident_history'])}")

---
## Cell 7 — CRUD: Read operations

In [ ]:
print('=== READ 1: Find customer C0182 (full case) ===')
doc = customer_cases.find_one(
    {'customer_id': 'C0182'},
    {'customer_id': 1, 'home_zone': 1, 'customer_type': 1,
     'account_status': 1, 'complaints': 1, '_id': 0}
)
print(f"  Customer: {doc['customer_id']} | Zone: {doc['home_zone']} | Status: {doc['account_status']}")
print(f"  Complaints: {len(doc['complaints'])}")
for c in doc['complaints']:
    print(f"    {c['complaint_id']}: {c['complaint_type']} — {c['status']} (£{c['compensation_amount']})")

print('\n=== READ 2: All High-severity Open complaints in North zone ===')
north_high = list(customer_cases.find(
    {
        'home_zone': 'North',
        'complaints': {'$elemMatch': {'severity': 'High', 'status': 'Open'}}
    },
    {'customer_id': 1, 'complaints': 1, '_id': 0}
))
print(f"  Found {len(north_high)} North zone customers with High/Open complaints")

print('\n=== READ 3: InRepair vehicles with battery below 70% ===')
at_risk = list(fleet_health.find(
    {'maintenance_status': 'InRepair', 'battery_health_pct': {'$lt': 70}},
    {'vehicle_id': 1, 'battery_health_pct': 1, 'assigned_zone': 1, '_id': 0}
))
print(f"  Found {len(at_risk)} at-risk vehicles:")
for v in at_risk[:5]:
    print(f"    {v['vehicle_id']}: {v['battery_health_pct']}% battery — {v['assigned_zone']}")

## Cell 8 — CRUD: Update operations

In [ ]:
print('=== UPDATE 1: Resolve complaint CP0295 for customer C0182 ===')

# Before
before = customer_cases.find_one(
    {'customer_id': 'C0182', 'complaints.complaint_id': 'CP0295'},
    {'complaints.$': 1, '_id': 0}
)
print(f"  Before: status = {before['complaints'][0]['status']}, resolution_days = {before['complaints'][0]['resolution_days']}")

# Update
customer_cases.update_one(
    {'customer_id': 'C0182', 'complaints.complaint_id': 'CP0295'},
    {'$set': {
        'complaints.$.status':          'Resolved',
        'complaints.$.resolution_days': 5
    }}
)

# After
after = customer_cases.find_one(
    {'customer_id': 'C0182', 'complaints.complaint_id': 'CP0295'},
    {'complaints.$': 1, '_id': 0}
)
print(f"  After:  status = {after['complaints'][0]['status']}, resolution_days = {after['complaints'][0]['resolution_days']}")

print('\n=== UPDATE 2: Update battery health for V002 after telematics reading ===')

before_bat = fleet_health.find_one({'vehicle_id': 'V002'}, {'battery_health_pct': 1, '_id': 0})
print(f"  Before: battery_health_pct = {before_bat['battery_health_pct']}")

fleet_health.update_one(
    {'vehicle_id': 'V002'},
    {'$set': {'battery_health_pct': 65.3}}
)

after_bat = fleet_health.find_one({'vehicle_id': 'V002'}, {'battery_health_pct': 1, '_id': 0})
print(f"  After:  battery_health_pct = {after_bat['battery_health_pct']}")

## Cell 9 — CRUD: Delete operation

In [ ]:
print('=== DELETE: Insert and then remove a test document ===')

# Insert test document
customer_cases.insert_one({
    'customer_id':   'TEST_001',
    'home_zone':     'North',
    'customer_type': 'Test',
    'account_status': 'Inactive',
    'orders': [], 'complaints': [], 'app_events': []
})
print(f"  Inserted TEST_001")
print(f"  Exists before delete: {customer_cases.find_one({'customer_id': 'TEST_001'}) is not None}")

# Delete
result = customer_cases.delete_one({'customer_id': 'TEST_001'})
print(f"  Deleted {result.deleted_count} document(s)")
print(f"  Exists after delete:  {customer_cases.find_one({'customer_id': 'TEST_001'}) is not None}")

---
## Cell 10 — Aggregation Pipeline 1: Complaint count by zone and severity

In [ ]:
pipeline_1 = [
    # Unwind complaints array
    {'$unwind': '$complaints'},

    # Filter to Open or Escalated complaints only
    {'$match': {
        'complaints.status': {'$in': ['Open', 'Escalated']}
    }},

    # Group by zone and severity
    {'$group': {
        '_id': {
            'zone':     '$home_zone',
            'severity': '$complaints.severity'
        },
        'complaint_count':     {'$sum': 1},
        'total_compensation':  {'$sum': '$complaints.compensation_amount'},
        'avg_resolution_days': {'$avg': '$complaints.resolution_days'}
    }},

    # Sort by complaint count descending
    {'$sort': {'complaint_count': -1}},

    # Reshape output
    {'$project': {
        'zone':                '$_id.zone',
        'severity':            '$_id.severity',
        'complaint_count':     1,
        'total_compensation':  {'$round': ['$total_compensation', 2]},
        'avg_resolution_days': {'$round': ['$avg_resolution_days', 1]},
        '_id': 0
    }}
]

results_1 = list(customer_cases.aggregate(pipeline_1))
print('Complaint count by zone and severity (Open/Escalated only):')
print(f'{"Zone":<12} {"Severity":<12} {"Count":<8} {"Total Comp":<14} {"Avg Days"}')
print('-' * 55)
for r in results_1[:10]:
    print(f"{r['zone']:<12} {r['severity']:<12} {r['complaint_count']:<8} £{r['total_compensation']:<13} {r['avg_resolution_days']}")

## Cell 11 — Aggregation Pipeline 2: Fleet risk summary by zone

In [ ]:
pipeline_2 = [
    # Unwind incident history (preserve vehicles with no incidents)
    {'$unwind': {'path': '$incident_history', 'preserveNullAndEmptyArrays': True}},

    # Group by zone and maintenance status
    {'$group': {
        '_id': {
            'zone':               '$assigned_zone',
            'maintenance_status': '$maintenance_status'
        },
        'vehicle_count':      {'$sum': 1},
        'avg_battery_health': {'$avg': '$battery_health_pct'},
        'critical_incidents': {
            '$sum': {
                '$cond': [
                    {'$in': ['$incident_history.incident_type', ['BatteryAlert', 'VehicleFault']]},
                    1, 0
                ]
            }
        }
    }},

    {'$sort': {'_id.zone': 1, '_id.maintenance_status': 1}},

    {'$project': {
        'zone':               '$_id.zone',
        'maintenance_status': '$_id.maintenance_status',
        'vehicle_count':      1,
        'avg_battery_health': {'$round': ['$avg_battery_health', 1]},
        'critical_incidents': 1,
        '_id': 0
    }}
]

results_2 = list(fleet_health.aggregate(pipeline_2))
print('Fleet risk summary by zone and maintenance status:')
print(f'{"Zone":<12} {"Status":<12} {"Vehicles":<10} {"Avg Battery":<14} {"Critical Inc"}')
print('-' * 58)
for r in results_2:
    print(f"{r['zone']:<12} {r['maintenance_status']:<12} {r['vehicle_count']:<10} {r['avg_battery_health']:<14} {r['critical_incidents']}")

## Cell 12 — Collection summary

In [ ]:
print('=== northstar_db collection summary ===')
for name in db.list_collection_names():
    count = db[name].count_documents({})
    print(f'  {name}: {count} documents')

client.close()
print('\nConnection closed.')